# FlowCast: Deep Learning-Based Short-Horizon Traffic Flow Forecasting
**Problem Statement**: L2 — Smart Traffic Management System for Urban Congestion
**Dataset**: UCI Machine Learning Repository (ID: 734) - 36 Loop Sensors along Northern Virginia Arterials
**Horizon**: 15-Minute Forward Volume Forecasting + Congestion Risk Classification

In [ ]:
# Install and import dependencies
!pip install -q torch pandas scikit-learn matplotlib seaborn scipy

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.load_data import load_raw_traffic_data
from src.data.preprocess import prepare_data_splits
from src.models.baseline import FlowCastBaseline
from src.models.gru_model import FlowCastGRU, get_default_device
from src.evaluation.metrics import evaluate_model_performance

print("PyTorch Version:", torch.__version__)
print("Accelerator Device:", get_default_device())

## 1. Exploratory Data Analysis (EDA) and Preprocessing
We load the official UCI dataset (2,101 timestamps) and partition chronologically into:
- **Train**: 1,061 quarter-hours
- **Val**: 200 quarter-hours
- **Test**: 840 quarter-hours

In [ ]:
splits = prepare_data_splits(val_size=200)
print("Train seq shape:", splits.X_train_seq.shape)
print("Val seq shape:", splits.X_val_seq.shape)
print("Test seq shape:", splits.X_test_seq.shape)
print("Derived non-leaking thresholds:", splits.thresholds)

# Plot target distribution
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(splits.Y_train.flatten(), bins=50, kde=True, ax=ax, color="#1f77b4")
ax.axvline(splits.thresholds[0], color="green", linestyle="--", label=f"Low/Moderate ({splits.thresholds[0]:.3f})")
ax.axvline(splits.thresholds[1], color="red", linestyle="--", label=f"Moderate/High ({splits.thresholds[1]:.3f})")
ax.set_title("Traffic Volume Target Distribution (Training Data)")
ax.legend()
plt.show()

## 2. Train Random Forest Baseline
Evaluating conventional ML baseline on flattened features across all 36 sensors.

In [ ]:
rf = FlowCastBaseline(n_estimators=100, max_depth=16, random_state=42)
rf.fit(splits.X_train_flat, splits.Y_train)
rf_preds = rf.predict(splits.X_test_flat)
rf_metrics = evaluate_model_performance(splits.Y_test, rf_preds, splits.thresholds, "Random Forest")
print("Baseline Regression:", rf_metrics["regression"])
print("Baseline Classification:", rf_metrics["classification"]["macro_f1"])

## 3. Train FlowCast Spatio-Temporal GRU
Dual-branch neural network consuming genuine 10-step sequence with auxiliary context.

In [ ]:
from src.models.train import train_gru
device = get_default_device()
gru_model, train_losses, val_losses = train_gru(splits, device=device, epochs=35, batch_size=32)

test_seq = torch.tensor(splits.X_test_seq, dtype=torch.float32).to(device)
test_ctx = torch.tensor(splits.X_test_ctx, dtype=torch.float32).to(device)
with torch.no_grad():
    gru_preds = gru_model(test_seq, test_ctx).cpu().numpy()

gru_metrics = evaluate_model_performance(splits.Y_test, gru_preds, splits.thresholds, "FlowCast GRU")
print("GRU Regression:", gru_metrics["regression"])
print("GRU Classification Macro-F1:", gru_metrics["classification"]["macro_f1"])

## 4. Evaluation and Visualizations
Plotting actual vs predicted and comparison between models.

In [ ]:
from src.evaluation.plots import plot_actual_vs_predicted, plot_training_curves
plot_training_curves(train_losses, val_losses)
plot_actual_vs_predicted(splits.Y_test, gru_preds, sensor_id=0)
plt.show()